In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
#Open the csv of the dataset
file_name = "MIMIC-IV-Echonext_final_paired_df_with_demographics.csv"
df = pd.read_csv(file_name)
df.head(25)


/tmp/ipykernel_367214/3681657828.py:3: DtypeWarning: Columns (46,48,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_name)


,study_id,subject_id,file_name,ecg_time,path,measurement_id,measurement_datetime,aortic_regurg,aortic_stenosis,biplane_lvef,...,ventricular_rate,pr_interval,qrs_duration,qt_corrected,atrial_rate,qt_interval,rr_interval_sec,atrial_rate_proxy,sex,age_at_ecg
0,47218930,10000764,47218930,2132-10-14 19:40:00,files/p1000/p10000764/s47218930/47218930,6335876,2132-10-17 10:27:00,Mild [1+],Mild (>1.5cm2),NaN,...,90.0,202.0,86.0,446.0,NaN,364.0,0.666,90.0,male,86.0
1,44069449,10000764,44069449,2132-10-15 07:12:00,files/p1000/p10000764/s44069449/44069449,6335876,2132-10-17 10:27:00,Mild [1+],Mild (>1.5cm2),NaN,...,123.0,194.0,78.0,504.0,NaN,352.0,0.487,123.0,male,86.0
2,44837313,10000764,44837313,2132-10-15 08:39:00,files/p1000/p10000764/s44837313/44837313,6335876,2132-10-17 10:27:00,Mild [1+],Mild (>1.5cm2),NaN,...,113.0,164.0,86.0,503.0,NaN,366.0,0.530,113.0,male,86.0
3,40539087,10000764,40539087,2132-10-16 09:54:00,files/p1000/p10000764/s40539087/40539087,6335876,2132-10-17 10:27:00,Mild [1+],Mild (>1.5cm2),NaN,...,75.0,222.0,82.0,429.0,NaN,384.0,0.800,75.0,male,86.0
4,43681375,10000764,43681375,2132-10-17 08:31:00,files/p1000/p10000764/s43681375/43681375,6335876,2132-10-17 10:27:00,Mild [1+],Mild (>1.5cm2),NaN,...,90.0,206.0,82.0,424.0,NaN,346.0,0.666,90.0,male,86.0
5,48575806,10000898,48575806,2187-09-24 13:26:00,files/p1000/p10000898/s48575806/48575806,5239624,2187-10-04 09:00:00,NaN,NaN,NaN,...,78.0,180.0,86.0,442.0,NaN,388.0,0.769,78.0,female,79.0
6,47499371,10000980,47499371,2187-02-21 01:21:00,files/p1000/p10000980/s47499371/47499371,6545834,2188-01-04 10:57:00,Trace,NaN,NaN,...,58.0,186.0,80.0,462.0,NaN,470.0,1.034,58.0,female,74.0
7,42546971,10000980,42546971,2188-01-03 12:33:00,files/p1000/p10000980/s42546971/42546971,6545834,2188-01-04 10:57:00,Trace,NaN,NaN,...,57.0,168.0,84.0,472.0,NaN,484.0,1.052,57.0,female,75.0
8,47004256,10000980,47004256,2188-01-03 19:56:00,files/p1000/p10000980/s47004256/47004256,6545834,2188-01-04 10:57:00,Trace,NaN,NaN,...,67.0,164.0,80.0,484.0,NaN,458.0,0.895,67.0,female,75.0
9,48376834,10000980,48376834,2189-06-27 06:43:00,files/p1000/p10000980/s48376834/48376834,6870164,2189-06-27 14:51:00,NaN,NaN,NaN,...,84.0,164.0,84.0,473.0,NaN,400.0,0.714,84.0,female,76.0


In [6]:
#Lets split the datset (Creating the split column)
# Split the paired ECG-echo dataset at the patient level.
# Train patients keep all ECGs; validation and test patients keep only their most recent ECG.
# Rows from validation/test patients that are not the most recent ECG are marked as no_split.

split_df = df.copy()
split_df["subject_id"] = split_df["subject_id"].astype(str)
split_df["ecg_time"] = pd.to_datetime(split_df["ecg_time"], errors="coerce")
split_df["measurement_datetime"] = pd.to_datetime(split_df["measurement_datetime"], errors="coerce")
split_df = split_df.dropna(subset=["subject_id", "ecg_time"]).copy()
split_df = split_df.sort_values(["subject_id", "ecg_time", "measurement_datetime"]).reset_index(drop=True)
split_df["split"] = "no_split"

unique_patients = split_df["subject_id"].drop_duplicates().sample(frac=1.0, random_state=14).tolist()
n_patients = len(unique_patients)
n_train_patients = int(n_patients * 0.6)
n_val_patients = int(n_patients * 0.2)

train_patient_ids = set(unique_patients[:n_train_patients])
val_patient_ids = set(unique_patients[n_train_patients:n_train_patients + n_val_patients])
test_patient_ids = set(unique_patients[n_train_patients + n_val_patients:])

split_df.loc[split_df["subject_id"].isin(train_patient_ids), "split"] = "train"

val_latest_idx = (
    split_df[split_df["subject_id"].isin(val_patient_ids)]
    .sort_values(["subject_id", "ecg_time", "measurement_datetime"])
    .groupby("subject_id", as_index=False)
    .tail(1)
    .index
)
split_df.loc[val_latest_idx, "split"] = "val"

test_latest_idx = (
    split_df[split_df["subject_id"].isin(test_patient_ids)]
    .sort_values(["subject_id", "ecg_time", "measurement_datetime"])
    .groupby("subject_id", as_index=False)
    .tail(1)
    .index
)
split_df.loc[test_latest_idx, "split"] = "test"

assert set(split_df.loc[split_df["split"] == "train", "subject_id"]).isdisjoint(set(split_df.loc[split_df["split"] == "val", "subject_id"]))
assert set(split_df.loc[split_df["split"] == "train", "subject_id"]).isdisjoint(set(split_df.loc[split_df["split"] == "test", "subject_id"]))
assert set(split_df.loc[split_df["split"] == "val", "subject_id"]).isdisjoint(set(split_df.loc[split_df["split"] == "test", "subject_id"]))

print(f"Patients total: {n_patients}")
print(f"ECGs total: {len(split_df)}")
print(f"Patients split - train: {len(train_patient_ids)}, val: {len(val_patient_ids)}, test: {len(test_patient_ids)}")
print(f"ECGs split - train: {(split_df['split'] == 'train').sum()}, val: {(split_df['split'] == 'val').sum()}, test: {(split_df['split'] == 'test').sum()}, no_split: {(split_df['split'] == 'no_split').sum()}")
print(f"Unique patients by split - train: {split_df.loc[split_df['split'] == 'train', 'subject_id'].nunique()} / val: {split_df.loc[split_df['split'] == 'validation', 'subject_id'].nunique()} / test: {split_df.loc[split_df['split'] == 'test', 'subject_id'].nunique()}")

split_df.head()
#Save this df to a csv file
split_df.to_csv("MIMIC-IV-Echonext_final.csv", index=False)  


Patients total: 54432
ECGs total: 294568
Patients split - train: 32659, val: 10886, test: 10887
ECGs split - train: 176860, val: 10886, test: 10887, no_split: 95935
Unique patients by split - train: 32659 / val: 0 / test: 10887


# Now lets build the .npy files just as Echonext does

In [14]:
"""
Build EchoNext-mini-style waveform .npy files from MIMIC-IV-ECG WFDB records.

What this replicates from EchoNext-mini:
  - One waveform file per split: <PREFIX>_<split>_waveforms.npy
    shape = (N, 1, 2500, 12)   -> 10s @ 250Hz, 12 leads
  - Row i of the waveform array corresponds to row i of that split's
    metadata (the row order within the split is preserved from the CSV).
  - Splits are: train, val, test, no_split (only the ones present in your CSV).
  - Optional preprocessing matching the EchoNext README:
        1. Median-filtered per lead (baseline wander removal)
        2. Clipped at the 0.1st / 99.9th percentile (per lead)
        3. Normalized using dataset-wide mean and standard deviation (per lead)
    The stats for steps 2-3 are computed ONCE from a reference split (train,
    to avoid leakage) and saved to a JSON file for reproducibility, then
    reused for every split (including train itself).

MIMIC-IV-ECG source records are WFDB (.hea/.dat), 12-lead, 10s, 500 Hz
(5000 samples/lead), so we downsample 500 -> 250 Hz to match.

Requirements:
    pip install wfdb numpy pandas scipy tqdm

Expected CSV columns (adjust CSV_PATH / column names below):
    - a "split" column with values in {"train", "val", "test", "no_split"}
    - a column with the path to the record *without* extension, e.g.
      "files/p1000/p10000764/s47218930/47218930"
"""

import os
import json
import wfdb
from scipy.signal import resample_poly, medfilt
from tqdm import tqdm

# ----------------------------------------------------------------------
# CONFIG - edit these for your setup
# ----------------------------------------------------------------------
CSV_PATH = "MIMIC-IV-Echonext_final.csv"         # tabular features CSV
RECORD_PATH_COL = "path"          # column with e.g. files/p1000/p10000764/s47218930/47218930
SPLIT_COL = "split"                      # column with train/val/test/no_split
MIMIC_ROOT = "/home/amendoza/datasets/mimic4ecg/" # directory that contains files/p1000/...
OUTPUT_DIR = "/media/amendoza/HDD/MIMICIV_ECG_echo/"
FILE_PREFIX = "MIMICIV_ECG_Echo"                # -> MyDataset_train_waveforms.npy, etc.

TARGET_FS = 250
TARGET_LEN = 2500      # 10s * 250Hz
SOURCE_FS = 500         # MIMIC-IV-ECG native sampling rate
# Canonical 12-lead order used by EchoNext / most 12-lead conventions
CANONICAL_LEADS = ["I", "II", "III", "aVR", "aVL", "aVF",
                    "V1", "V2", "V3", "V4", "V5", "V6"]
N_LEADS = len(CANONICAL_LEADS)

# Fixed split ordering used to group the single combined CSV, and to build
# the npy files. This is what makes "row i of <PREFIX>_train_waveforms.npy
# == row i of the train block in <PREFIX>_metadata.csv" true and unambiguous:
# the CSV is stable-sorted into contiguous split blocks in this order.
SPLIT_ORDER = ["train", "val", "test", "no_split"]

# --- Preprocessing config -----------------------------------------------

# Match EchoNext reference preprocessing:
# baseline = median_filter_0.6s(median_filter_0.2s(signal))
# filtered_signal = signal - baseline
APPLY_MEDIAN_FILTER = True
MEDIAN_FILTER_WINDOW_SEC_1 = 0.2
MEDIAN_FILTER_WINDOW_SEC_2 = 0.6

# EchoNext reconstructs the dependent limb leads from I and II
RECONSTRUCT_DEPENDENT_LIMB_LEADS = True

# Clip + normalize using statistics estimated from the training split
APPLY_PREPROCESSING = True

STATS_SPLIT = "train"
STATS_PATH = os.path.join(
    OUTPUT_DIR,
    f"{FILE_PREFIX}_waveform_stats.json"
)
COMPUTE_STATS = True

PERCENTILE_SAMPLE_SIZE = 20000
CLIP_LOW_PCTL = 0.1
CLIP_HIGH_PCTL = 99.9
RANDOM_SEED = 14

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ----------------------------------------------------------------------

def canonicalize_lead_name(name: str) -> str:
    """
    Normalize common variations in lead names to the convention used
    by EchoNext.
    """
    name = str(name).strip()

    aliases = {
        "AVR": "aVR",
        "AVL": "aVL",
        "AVF": "aVF",
        "avr": "aVR",
        "avl": "aVL",
        "avf": "aVF",
    }

    return aliases.get(name, name)


def _fill_partial_nans(x: np.ndarray):
    """
    Handle NaNs within one ECG lead.

    Returns
    -------
    x : np.ndarray or None
        Interpolated lead. None if the entire lead is missing.
    had_partial_nans : bool
        True if some (but not all) samples were NaN.
    """

    x = np.asarray(x, dtype=np.float64).copy()
    nans = np.isnan(x)

    if not nans.any():
        return x, False

    # Entire lead missing
    if nans.all():
        return None, False

    # Partial missing samples -> interpolate
    valid = ~nans

    x[nans] = np.interp(
        np.flatnonzero(nans),
        np.flatnonzero(valid),
        x[valid]
    )

    return x, True

def load_and_resample_ecg(record_path_no_ext: str):
    """
    Load and prepare one MIMIC-IV-ECG record while preserving dataset
    row alignment.

    The function NEVER intentionally drops a record.

    Handling of problematic ECGs:
      - Partial NaNs within a lead -> linear interpolation
      - Missing limb leads -> attempt recovery of I/II from available
        limb leads using least squares
      - Missing precordial lead -> zero-impute temporarily
      - Unreadable / severely malformed ECG -> all-zero placeholder

    Imputed leads are later forced to zero AFTER normalization so that
    zero represents the training-set mean.

    Returns
    -------
    sig : np.ndarray
        Shape (2500, 12)

    quality : dict
        Information about interpolation/imputation.
    """

    quality = {
        "record_path": record_path_no_ext,
        "read_failed": False,
        "partial_nan_leads": [],
        "imputed_leads": [],
        "limb_leads_recovered": False,
        "length_adjusted": False,
        "original_length_after_resampling": TARGET_LEN,
        "padding_start": None,
        "whole_record_imputed": False,
        "usable_for_stats": True,
    }

    # ==========================================================
    # READ RECORD
    # ==========================================================

    try:
        record = wfdb.rdrecord(record_path_no_ext)

    except Exception as e:

        print(
            f"[WARN] failed to read {record_path_no_ext}: {e}. "
            f"Using neutral ECG placeholder."
        )

        quality["read_failed"] = True
        quality["whole_record_imputed"] = True
        quality["usable_for_stats"] = False
        quality["imputed_leads"] = CANONICAL_LEADS.copy()

        sig = np.zeros(
            (TARGET_LEN, N_LEADS),
            dtype=np.float32
        )

        return sig, quality

    raw_sig = np.asarray(
        record.p_signal,
        dtype=np.float64
    )

    sig_names = [
        canonicalize_lead_name(x)
        for x in record.sig_name
    ]

    # ==========================================================
    # READ AVAILABLE LEADS
    # ==========================================================

    lead_data = {}

    for lead in CANONICAL_LEADS:

        if lead not in sig_names:
            lead_data[lead] = None
            continue

        idx = sig_names.index(lead)

        x, had_partial_nan = _fill_partial_nans(
            raw_sig[:, idx]
        )

        lead_data[lead] = x

        if had_partial_nan:
            quality["partial_nan_leads"].append(lead)

    # ==========================================================
    # RECOVER I AND II FROM AVAILABLE LIMB LEADS
    #
    # All standard limb leads can be written:
    #
    # I   =  1*I + 0*II
    # II  =  0*I + 1*II
    # III = -1*I + 1*II
    # aVR = -.5I -.5II
    # aVL =  1*I -.5II
    # aVF = -.5I + 1*II
    #
    # Therefore, if I or II is absent but enough other limb
    # leads exist, solve for I/II using least squares.
    # ==========================================================

    limb_coefficients = {
        "I":   [ 1.0,  0.0],
        "II":  [ 0.0,  1.0],
        "III": [-1.0,  1.0],
        "aVR": [-0.5, -0.5],
        "aVL": [ 1.0, -0.5],
        "aVF": [-0.5,  1.0],
    }

    limb_available = [
        lead
        for lead in limb_coefficients
        if lead_data.get(lead) is not None
    ]

    need_limb_recovery = (
        lead_data.get("I") is None
        or lead_data.get("II") is None
    )

    if need_limb_recovery:

        A = np.array(
            [limb_coefficients[l] for l in limb_available],
            dtype=np.float64
        )

        # Need two linearly independent equations
        if (
            len(limb_available) >= 2
            and np.linalg.matrix_rank(A) >= 2
        ):

            Y = np.column_stack(
                [lead_data[l] for l in limb_available]
            )

            # Solve simultaneously for every time point
            solution, _, _, _ = np.linalg.lstsq(
                A,
                Y.T,
                rcond=None
            )

            recovered_I = solution[0]
            recovered_II = solution[1]

            if lead_data.get("I") is None:
                lead_data["I"] = recovered_I
                quality["imputed_leads"].append("I")

            if lead_data.get("II") is None:
                lead_data["II"] = recovered_II
                quality["imputed_leads"].append("II")

            quality["limb_leads_recovered"] = True

        else:

            # There is not enough information to reconstruct a
            # physiologically meaningful limb system.
            #
            # Preserve alignment but use a neutral ECG placeholder.
            print(
                f"[WARN] {record_path_no_ext}: insufficient limb "
                f"leads to recover I/II. Using neutral ECG placeholder."
            )

            quality["whole_record_imputed"] = True
            quality["usable_for_stats"] = False
            quality["imputed_leads"] = CANONICAL_LEADS.copy()

            sig = np.zeros(
                (TARGET_LEN, N_LEADS),
                dtype=np.float32
            )

            return sig, quality

    # ==========================================================
    # RECONSTRUCT DEPENDENT LIMB LEADS EXACTLY LIKE EchoNext
    # ==========================================================

    I = lead_data["I"]
    II = lead_data["II"]

    III = II - I
    aVR = -(I + II) / 2.0
    aVF = II - I / 2.0
    aVL = I - II / 2.0

    # ==========================================================
    # PRECORDIAL LEADS
    # ==========================================================

    chest = {}

    for lead in [
        "V1", "V2", "V3",
        "V4", "V5", "V6"
    ]:

        if lead_data.get(lead) is None:

            print(
                f"[WARN] {record_path_no_ext}: {lead} completely "
                f"missing. Imputing neutral lead."
            )

            chest[lead] = np.zeros_like(I)

            quality["imputed_leads"].append(lead)
            quality["usable_for_stats"] = False

        else:

            chest[lead] = lead_data[lead]

    # ==========================================================
    # CANONICAL EchoNext ORDER
    # ==========================================================

    sig = np.column_stack([
        I,
        II,
        III,
        aVR,
        aVL,
        aVF,
        chest["V1"],
        chest["V2"],
        chest["V3"],
        chest["V4"],
        chest["V5"],
        chest["V6"],
    ])

    # ==========================================================
    # DOWNSAMPLING
    # ==========================================================

    fs = float(
        record.fs if record.fs else SOURCE_FS
    )

    if fs == 500 and TARGET_FS == 250:

        # Match EchoNext
        sig = sig[::2]

    elif fs != TARGET_FS:

        print(
            f"[WARN] unexpected sampling rate {fs} Hz "
            f"for {record_path_no_ext}; using resample_poly"
        )

        sig = resample_poly(
            sig,
            TARGET_FS,
            int(fs),
            axis=0
        )

    # ==========================================================
    # LENGTH HANDLING
    #
    # Preserve the row instead of dropping it.
    # ==========================================================

    n = sig.shape[0]

    quality["original_length_after_resampling"] = int(n)

    if n < TARGET_LEN:

        quality["length_adjusted"] = True
        quality["usable_for_stats"] = False
        quality["padding_start"] = int(n)

        print(
            f"[WARN] {record_path_no_ext}: only {n} samples "
            f"after resampling; zero-padding to {TARGET_LEN}."
        )

        pad = np.zeros(
            (TARGET_LEN - n, N_LEADS),
            dtype=sig.dtype
        )

        sig = np.vstack([
            sig,
            pad
        ])

    elif n > TARGET_LEN:

        quality["length_adjusted"] = True
        quality["usable_for_stats"] = False

        print(
            f"[WARN] {record_path_no_ext}: {n} samples "
            f"after resampling; truncating to {TARGET_LEN}."
        )

        sig = sig[:TARGET_LEN]

    return sig.astype(np.float32), quality

# ----------------------------------------------------------------------
def apply_median_filter(
    sig: np.ndarray,
    fs: int = TARGET_FS
) -> np.ndarray:
    """
    EchoNext-style baseline wander removal.

    The EchoNext reference implementation estimates the baseline
    sequentially:

        baseline_1 = median_filter(signal, 0.2 sec)
        baseline_2 = median_filter(baseline_1, 0.6 sec)

    and then:

        filtered = signal - baseline_2

    At 250 Hz:
        0.2 sec -> 51 samples
        0.6 sec -> 151 samples

    Parameters
    ----------
    sig : np.ndarray
        ECG waveform with shape (time, 12).
    fs : int
        Sampling frequency. Expected to be 250 Hz here.

    Returns
    -------
    np.ndarray
        Baseline-corrected ECG with the same shape as input.
    """

    sig = np.asarray(sig)

    if sig.ndim != 2:
        raise ValueError(
            f"Expected 2D ECG array (time, leads), "
            f"received shape {sig.shape}"
        )

    # EchoNext formula:
    # int(round(seconds * sampling_frequency)) + 1
    window_1 = int(np.round(
        MEDIAN_FILTER_WINDOW_SEC_1 * fs
    )) + 1

    window_2 = int(np.round(
        MEDIAN_FILTER_WINDOW_SEC_2 * fs
    )) + 1

    # scipy.signal.medfilt requires odd kernel sizes.
    # At 250 Hz these are already 51 and 151.
    if window_1 % 2 == 0:
        window_1 += 1

    if window_2 % 2 == 0:
        window_2 += 1

    out = np.empty_like(sig)

    for c in range(sig.shape[1]):

        # First median filter: 0.2 s
        baseline = medfilt(
            sig[:, c],
            kernel_size=window_1
        )

        # Second sequential median filter: 0.6 s
        baseline = medfilt(
            baseline,
            kernel_size=window_2
        )

        # Remove estimated baseline
        out[:, c] = sig[:, c] - baseline

    return out


def apply_clip_and_normalize(sig: np.ndarray, stats: dict) -> np.ndarray:
    """
    sig: (T, 12). stats holds per-lead p_low/p_high/mean/std, each length 12.
    """
    p_low = np.asarray(stats["p_low"], dtype=np.float32)
    p_high = np.asarray(stats["p_high"], dtype=np.float32)
    mean = np.asarray(stats["mean"], dtype=np.float32)
    std = np.asarray(stats["std"], dtype=np.float32)
    std_safe = np.where(std == 0, 1.0, std)

    sig = np.clip(sig, p_low, p_high)   # broadcasts over the lead axis
    sig = (sig - mean) / std_safe
    return sig.astype(np.float32)

def process_signal(
    record_path_no_ext: str,
    stats: dict | None = None
):
    """
    Full processing pipeline.

    Returns
    -------
    sig : np.ndarray
        Shape (2500, 12)

    quality : dict
        Quality / imputation information.
    """

    sig, quality = load_and_resample_ecg(
        record_path_no_ext
    )

    # Entire record is missing:
    # do not median-filter meaningless zeros.
    if not quality["whole_record_imputed"]:

        if APPLY_MEDIAN_FILTER:
            sig = apply_median_filter(sig)

        if APPLY_PREPROCESSING and stats is not None:
            sig = apply_clip_and_normalize(
                sig,
                stats
            )

    else:

        # Zero in standardized space = training-set mean
        sig = np.zeros(
            (TARGET_LEN, N_LEADS),
            dtype=np.float32
        )

    # ======================================================
    # Force completely imputed leads to zero AFTER
    # standardization.
    #
    # This is important:
    # raw 0 -> (0 - mean)/std is not necessarily 0.
    #
    # Setting it here means "neutral / training mean".
    # ======================================================

    for lead in quality["imputed_leads"]:

        if lead in CANONICAL_LEADS:

            idx = CANONICAL_LEADS.index(lead)

            sig[:, idx] = 0.0

    # ======================================================
    # Neutralize zero-padded time samples after normalization
    # ======================================================

    if quality["padding_start"] is not None:

        start = quality["padding_start"]

        sig[start:, :] = 0.0

    return sig.astype(np.float32), quality


# ----------------------------------------------------------------------
def compute_dataset_stats(df_stats: pd.DataFrame) -> dict:
    """
    Computes per-lead clip percentiles + mean/std, in two passes:
      Pass 1 (percentiles): on a (possibly subsampled) set of records,
        after resampling + median filtering, but BEFORE clipping/normalizing.
      Pass 2 (mean/std): streaming over ALL records in df_stats, using the
        percentiles from pass 1 to clip before accumulating sum/sumsq.
    Returns a dict with p_low, p_high, mean, std (each length N_LEADS),
    plus bookkeeping metadata.
    """
    rng = np.random.default_rng(RANDOM_SEED)

    # ---------- Pass 1: percentiles ----------
    if PERCENTILE_SAMPLE_SIZE is not None and PERCENTILE_SAMPLE_SIZE < len(df_stats):
        sample_idx = rng.choice(len(df_stats), size=PERCENTILE_SAMPLE_SIZE, replace=False)
        df_pctl = df_stats.iloc[sample_idx]
    else:
        df_pctl = df_stats

    per_lead_values = [[] for _ in range(N_LEADS)]
    for row in tqdm(df_pctl.itertuples(index=False), total=len(df_pctl),
                     desc="stats pass 1/2 (percentiles)"):
        rel_path = getattr(row, RECORD_PATH_COL)
        full_path = os.path.join(MIMIC_ROOT, rel_path)

        sig, quality = load_and_resample_ecg(full_path)

        if not quality["usable_for_stats"]:
            continue
        if APPLY_MEDIAN_FILTER:
            sig = apply_median_filter(sig)

        for c in range(N_LEADS):
            per_lead_values[c].append(sig[:, c])

    p_low = np.zeros(N_LEADS, dtype=np.float64)
    p_high = np.zeros(N_LEADS, dtype=np.float64)
    for c in range(N_LEADS):
        concat = np.concatenate(per_lead_values[c])
        p_low[c], p_high[c] = np.percentile(concat, [CLIP_LOW_PCTL, CLIP_HIGH_PCTL])
    del per_lead_values  # free memory before pass 2

    # ---------- Pass 2: mean/std of clipped signal, streaming over full split ----------
    count = 0
    sum_ = np.zeros(N_LEADS, dtype=np.float64)
    sumsq = np.zeros(N_LEADS, dtype=np.float64)

    for row in tqdm(df_stats.itertuples(index=False), total=len(df_stats),
                     desc="stats pass 2/2 (mean/std)"):
        rel_path = getattr(row, RECORD_PATH_COL)
        full_path = os.path.join(MIMIC_ROOT, rel_path)

        sig, quality = load_and_resample_ecg(full_path)

        if not quality["usable_for_stats"]:
            continue
        if sig is None:
            continue
        if APPLY_MEDIAN_FILTER:
            sig = apply_median_filter(sig)

        sig = np.clip(sig, p_low, p_high)
        sum_ += sig.sum(axis=0)
        sumsq += (sig ** 2).sum(axis=0)
        count += sig.shape[0]

    mean = sum_ / count
    var = sumsq / count - mean ** 2
    std = np.sqrt(np.maximum(var, 0))

    stats = {
        "leads": CANONICAL_LEADS,
        "p_low": p_low.tolist(),
        "p_high": p_high.tolist(),
        "mean": mean.tolist(),
        "std": std.tolist(),
        "clip_low_pctl": CLIP_LOW_PCTL,
        "clip_high_pctl": CLIP_HIGH_PCTL,
        "median_filter_applied": APPLY_MEDIAN_FILTER,
        "median_filter_window_sec_1": MEDIAN_FILTER_WINDOW_SEC_1,
        "median_filter_window_sec_2": MEDIAN_FILTER_WINDOW_SEC_2,
        "target_fs": TARGET_FS,
        "stats_split": STATS_SPLIT,
        "n_records_used_for_percentiles": len(df_pctl),
        "n_records_used_for_mean_std": len(df_stats),
        "percentile_sample_size_config": PERCENTILE_SAMPLE_SIZE,
        "random_seed": RANDOM_SEED,
    }
    return stats


def save_stats(stats: dict, path: str):
    with open(path, "w") as f:
        json.dump(stats, f, indent=2)
    print(f"[DONE] waveform stats saved -> {path}")


def load_stats(path: str) -> dict:
    with open(path, "r") as f:
        return json.load(f)


# ----------------------------------------------------------------------
def build_split(
    df_split: pd.DataFrame,
    split_name: str,
    stats: dict | None
):
    """
    Build one waveform split WITHOUT ever changing the number
    or order of metadata rows.

    Any problematic ECG is retained using neutral imputation,
    and the event is written to a quality-control CSV.
    """

    n = len(df_split)

    out_path = os.path.join(
        OUTPUT_DIR,
        f"{FILE_PREFIX}_{split_name}_waveforms.npy"
    )

    quality_log_path = os.path.join(
        OUTPUT_DIR,
        f"{FILE_PREFIX}_{split_name}_waveform_quality_log.csv"
    )

    waveforms = np.lib.format.open_memmap(
        out_path,
        mode="w+",
        dtype=np.float32,
        shape=(n, 1, TARGET_LEN, N_LEADS)
    )

    quality_rows = []

    n_with_issues = 0

    for i, row in enumerate(
        tqdm(
            df_split.itertuples(index=False),
            total=n,
            desc=f"split={split_name}"
        )
    ):

        rel_path = getattr(
            row,
            RECORD_PATH_COL
        )

        full_path = os.path.join(
            MIMIC_ROOT,
            rel_path
        )

        sig, quality = process_signal(
            full_path,
            stats=stats
        )

        # Always write exactly one ECG for exactly one CSV row
        waveforms[i, 0] = sig

        has_issue = (
            quality["read_failed"]
            or quality["whole_record_imputed"]
            or quality["length_adjusted"]
            or len(quality["imputed_leads"]) > 0
            or len(quality["partial_nan_leads"]) > 0
        )

        if has_issue:

            n_with_issues += 1

            quality_rows.append({
                "split": split_name,
                "split_row_index": i,
                "record_path": rel_path,

                "read_failed":
                    quality["read_failed"],

                "whole_record_imputed":
                    quality["whole_record_imputed"],

                "partial_nan_leads":
                    ";".join(
                        quality["partial_nan_leads"]
                    ),

                "imputed_leads":
                    ";".join(
                        quality["imputed_leads"]
                    ),

                "limb_leads_recovered":
                    quality["limb_leads_recovered"],

                "length_adjusted":
                    quality["length_adjusted"],

                "original_length_after_resampling":
                    quality[
                        "original_length_after_resampling"
                    ],

                "padding_start":
                    quality["padding_start"],

                "usable_for_stats":
                    quality["usable_for_stats"],
            })

    waveforms.flush()
    del waveforms

    # ==========================================================
    # Save QC / imputation log
    # ==========================================================

    if quality_rows:

        quality_df = pd.DataFrame(
            quality_rows
        )

        quality_df.to_csv(
            quality_log_path,
            index=False
        )

        print(
            f"[QC] split={split_name}: "
            f"{n_with_issues}/{n} ECGs required "
            f"interpolation/imputation/length correction"
        )

        print(
            f"[QC] details saved -> "
            f"{quality_log_path}"
        )

    else:

        # Still write an empty log with useful columns
        pd.DataFrame(
            columns=[
                "split",
                "split_row_index",
                "record_path",
                "read_failed",
                "whole_record_imputed",
                "partial_nan_leads",
                "imputed_leads",
                "limb_leads_recovered",
                "length_adjusted",
                "original_length_after_resampling",
                "padding_start",
                "usable_for_stats",
            ]
        ).to_csv(
            quality_log_path,
            index=False
        )

        print(
            f"[QC] split={split_name}: "
            f"no problematic ECGs detected"
        )

    return df_split


# ----------------------------------------------------------------------
df = pd.read_csv(CSV_PATH)

if SPLIT_COL not in df.columns:
    raise ValueError(f"Expected a '{SPLIT_COL}' column in {CSV_PATH}")

unknown = set(df[SPLIT_COL].unique()) - set(SPLIT_ORDER)
if unknown:
    raise ValueError(f"Unexpected values in '{SPLIT_COL}': {unknown}. "
                        f"Expected a subset of {SPLIT_ORDER}")

# ---- Sort into contiguous split blocks (stable: preserves each split's
# original relative row order). This is what makes a SINGLE combined CSV
# line up with the separate per-split npy files, the same way EchoNext's
# single echonext_metadata_100k.csv lines up with its 4 npy files. ----
df[SPLIT_COL] = pd.Categorical(df[SPLIT_COL], categories=SPLIT_ORDER, ordered=True)
df = df.sort_values(SPLIT_COL, kind="stable").reset_index(drop=True)
df[SPLIT_COL] = df[SPLIT_COL].astype(str)

splits_present = [s for s in SPLIT_ORDER if s in df[SPLIT_COL].unique()]

# ---- Stats (used only if APPLY_PREPROCESSING is True) ----
stats = None
if APPLY_PREPROCESSING:
    if COMPUTE_STATS:
        if STATS_SPLIT not in splits_present:
            raise ValueError(
                f"STATS_SPLIT='{STATS_SPLIT}' not found among splits {splits_present}"
            )
        df_stats_split = df[df[SPLIT_COL] == STATS_SPLIT].reset_index(drop=True)
        stats = compute_dataset_stats(df_stats_split)
        save_stats(stats, STATS_PATH)
    elif os.path.exists(STATS_PATH):
        stats = load_stats(STATS_PATH)
        print(f"[INFO] loaded existing stats from {STATS_PATH}")
    else:
        raise FileNotFoundError(
            f"APPLY_PREPROCESSING=True and COMPUTE_STATS=False, but "
            f"{STATS_PATH} does not exist. Either set COMPUTE_STATS=True "
            f"once, or point STATS_PATH at an existing stats file."
        )

# ---- Build each split's npy off its contiguous slice of the sorted df,
# then stitch the (possibly-compacted) per-split frames back into ONE
# combined CSV, in the same split order. ----
final_frames = []
split_boundaries = {}
row_cursor = 0

for split_name in splits_present:
    df_split = df[df[SPLIT_COL] == split_name].reset_index(drop=True)

    df_split_final = build_split(df_split, split_name, stats=stats)
    n = len(df_split_final)

    split_boundaries[split_name] = {"start": row_cursor, "end": row_cursor + n}
    row_cursor += n
    final_frames.append(df_split_final)

    print(f"[DONE] {split_name}: "
            f"{FILE_PREFIX}_{split_name}_waveforms.npy ({n} x 1 x {TARGET_LEN} x 12)")

# ---- Single combined CSV, split-grouped, row-aligned with each npy ----
combined_df = pd.concat(final_frames, ignore_index=True)
meta_out = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX}_metadata.csv")
combined_df.to_csv(meta_out, index=False)

boundaries_out = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX}_split_boundaries.json")
with open(boundaries_out, "w") as f:
    json.dump(split_boundaries, f, indent=2)

print(f"[DONE] combined metadata ({len(combined_df)} rows) -> {meta_out}")
print(f"[DONE] split row-ranges within that CSV -> {boundaries_out}")
print(f"       {split_boundaries}")
print("       (df[df.split=='train'] on the combined CSV, or "
        "combined_df.iloc[start:end] using the boundaries file, both "
        "give you the exact rows matching <PREFIX>_train_waveforms.npy, etc.)")


/tmp/ipykernel_367214/1379752599.py:914: DtypeWarning: Columns (46,48,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)
stats pass 2/2 (mean/std): 100%|██████████| 176860/176860 [16:48<00:00, 175.40it/s]


[DONE] waveform stats saved -> /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_waveform_stats.json


split=train: 100%|██████████| 176860/176860 [15:08<00:00, 194.68it/s]


[QC] split=train: 2565/176860 ECGs required interpolation/imputation/length correction
[QC] details saved -> /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_train_waveform_quality_log.csv
[DONE] train: MIMICIV_ECG_Echo_train_waveforms.npy (176860 x 1 x 2500 x 12)


split=val: 100%|██████████| 10886/10886 [00:53<00:00, 203.70it/s]


[QC] split=val: 125/10886 ECGs required interpolation/imputation/length correction
[QC] details saved -> /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_val_waveform_quality_log.csv
[DONE] val: MIMICIV_ECG_Echo_val_waveforms.npy (10886 x 1 x 2500 x 12)


split=test: 100%|██████████| 10887/10887 [00:53<00:00, 203.91it/s]


[QC] split=test: 153/10887 ECGs required interpolation/imputation/length correction
[QC] details saved -> /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_test_waveform_quality_log.csv
[DONE] test: MIMICIV_ECG_Echo_test_waveforms.npy (10887 x 1 x 2500 x 12)


split=no_split: 100%|██████████| 95935/95935 [07:47<00:00, 205.23it/s]


[QC] split=no_split: 1428/95935 ECGs required interpolation/imputation/length correction
[QC] details saved -> /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_no_split_waveform_quality_log.csv
[DONE] no_split: MIMICIV_ECG_Echo_no_split_waveforms.npy (95935 x 1 x 2500 x 12)
[DONE] combined metadata (294568 rows) -> /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_metadata.csv
[DONE] split row-ranges within that CSV -> /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_split_boundaries.json
       {'train': {'start': 0, 'end': 176860}, 'val': {'start': 176860, 'end': 187746}, 'test': {'start': 187746, 'end': 198633}, 'no_split': {'start': 198633, 'end': 294568}}
       (df[df.split=='train'] on the combined CSV, or combined_df.iloc[start:end] using the boundaries file, both give you the exact rows matching <PREFIX>_train_waveforms.npy, etc.)


In [20]:
qc1 = pd.read_csv("MIMICIV_ECG_echo/MIMICIV_ECG_Echo_train_waveform_quality_log.csv")
qc2 = pd.read_csv("MIMICIV_ECG_echo/MIMICIV_ECG_Echo_val_waveform_quality_log.csv")
qc3 = pd.read_csv("MIMICIV_ECG_echo/MIMICIV_ECG_Echo_test_waveform_quality_log.csv")
qc4 = pd.read_csv("MIMICIV_ECG_echo/MIMICIV_ECG_Echo_no_split_waveform_quality_log.csv")
print(qc1["whole_record_imputed"].value_counts())
print(qc1["imputed_leads"].value_counts())
print("Any issue:", len(qc1))
print("Entire ECG imputed:", qc1["whole_record_imputed"].sum())

print(qc2["whole_record_imputed"].value_counts())
print(qc2["imputed_leads"].value_counts())
print("Any issue:", len(qc2))
print("Entire ECG imputed:", qc2["whole_record_imputed"].sum())

print(qc3["whole_record_imputed"].value_counts())
print(qc3["imputed_leads"].value_counts())
print("Any issue:", len(qc3))
print("Entire ECG imputed:", qc3["whole_record_imputed"].sum())

print(qc4["whole_record_imputed"].value_counts())
print(qc4["imputed_leads"].value_counts())
print("Any issue:", len(qc4))
print("Entire ECG imputed:", qc4["whole_record_imputed"].sum())


whole_record_imputed
False    2565
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
Any issue: 2565
Entire ECG imputed: 0
whole_record_imputed
False    125
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
Any issue: 125
Entire ECG imputed: 0
whole_record_imputed
False    153
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
Any issue: 153
Entire ECG imputed: 0
whole_record_imputed
False    1428
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
Any issue: 1428
Entire ECG imputed: 0


# NOw lets create the .npy for tabular features like EchoNext

In [10]:
"""
Build EchoNext-mini-style tabular_features.npy files from your metadata CSV.

What this replicates from EchoNext-mini:
  - One file per split: <PREFIX>_<split>_tabular_features.npy, shape (N, 7)
  - Column order: sex, ventricular_rate, atrial_rate, pr_interval,
                   qrs_duration, qt_corrected, age_at_ecg
  - Preprocessing (per the EchoNext README):
        * age_at_ecg capped at 90
        * sex binarized (female=0, male=1)
        * continuous features standardized (z-scored)
        * missing values imputed with the median, EXCEPT atrial_rate and
          pr_interval, which are imputed with 0
  - Stats (medians, mean/std) are computed ONCE from a reference split
    (train, to avoid leakage) and saved to a JSON file for reproducibility,
    then reused to transform every split.

IMPORTANT - run this AFTER build_echonext_style_waveforms.py, and point
CSV_PATH at ITS output (<PREFIX>_metadata.csv). That file is already
stable-sorted into contiguous split blocks and already has any
unreadable/dropped ECGs removed, so row i of the train block there is
guaranteed to be the same ECG as row i of <PREFIX>_train_waveforms.npy.
Running this on the raw, un-deduplicated CSV instead risks a row-count /
row-order mismatch against the waveform files.

Requirements:
    pip install numpy pandas
"""

# CONFIG 
CSV_PATH = "/media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_metadata.csv"  # output of the waveform script
SPLIT_COL = "split"
OUTPUT_DIR = "/media/amendoza/HDD/MIMICIV_ECG_echo/"
FILE_PREFIX = "MIMICIV_ECG_Echo"

# Source column names in your CSV
SEX_COL = "sex"                        # values like "male" / "female"
VENTRICULAR_RATE_COL = "ventricular_rate"
PR_INTERVAL_COL = "pr_interval"
QRS_DURATION_COL = "qrs_duration"
QT_CORRECTED_COL = "qt_corrected"
AGE_AT_ECG_COL = "age_at_ecg"

# --- Switch: where should "atrial_rate" come from? ---------------------
ATRIAL_RATE_SOURCE = "atrial_rate_proxy"     # "atrial_rate" or "atrial_rate_proxy"
ATRIAL_RATE_COL = "atrial_rate"
ATRIAL_RATE_PROXY_COL = "atrial_rate_proxy"

AGE_CAP = 90  # Like Echonext

SPLIT_ORDER = ["train", "val", "test", "no_split"]
STATS_SPLIT = "train"          # compute median/mean/std from this split only
STATS_PATH = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX}_tabular_stats.json")
COMPUTE_STATS = True           # False -> load an existing STATS_PATH instead

# Final output column order (matches Echonext)
FEATURE_COLS = ["sex", "ventricular_rate", "atrial_rate", "pr_interval",
                 "qrs_duration", "qt_corrected", "age_at_ecg"]
# Continuous features that get standardized (everything except sex)
CONTINUOUS_COLS = ["ventricular_rate", "atrial_rate", "pr_interval",
                    "qrs_duration", "qt_corrected", "age_at_ecg"]
# Missing values -> 0 (per EchoNext README, these two are special-cased)
ZERO_IMPUTE_COLS = ["atrial_rate", "pr_interval"]
# Missing values -> median (computed on STATS_SPLIT)
MEDIAN_IMPUTE_COLS = ["ventricular_rate", "qrs_duration", "qt_corrected", "age_at_ecg"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------------------------------
def _atrial_rate_source_col() -> str:
    if ATRIAL_RATE_SOURCE == "atrial_rate":
        return ATRIAL_RATE_COL
    elif ATRIAL_RATE_SOURCE == "atrial_rate_proxy":
        return ATRIAL_RATE_PROXY_COL
    else:
        raise ValueError(
            f"ATRIAL_RATE_SOURCE must be 'atrial_rate' or 'atrial_rate_proxy', "
            f"got {ATRIAL_RATE_SOURCE!r}"
        )


def extract_raw_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pulls the 7 source columns out of df into a fresh DataFrame with the
    canonical column names, applying only non-statistical transforms:
      - sex: "male"/"female" (any case) -> 1/0, else NaN
      - atrial_rate: taken from ATRIAL_RATE_COL or ATRIAL_RATE_PROXY_COL
        per ATRIAL_RATE_SOURCE
      - age_at_ecg: capped at AGE_CAP
      - everything else: coerced to float (non-numeric -> NaN)
    """
    out = pd.DataFrame(index=df.index)

    sex_raw = df[SEX_COL].astype(str).str.strip().str.lower()
    sex_map = {"male": 1.0, "m": 1.0, "1": 1.0, "1.0": 1.0,
               "female": 0.0, "f": 0.0, "0": 0.0, "0.0": 0.0}
    out["sex"] = sex_raw.map(sex_map)
    n_unmapped = out["sex"].isna().sum() - df[SEX_COL].isna().sum()
    if n_unmapped > 0:
        unrecognized = sorted(set(sex_raw[out["sex"].isna() & df[SEX_COL].notna()]))
        print(f"[WARN] {n_unmapped} rows had unrecognized '{SEX_COL}' values "
              f"{unrecognized}; treated as missing")

    out["ventricular_rate"] = pd.to_numeric(df[VENTRICULAR_RATE_COL], errors="coerce")

    atrial_col = _atrial_rate_source_col()
    if atrial_col not in df.columns:
        raise ValueError(f"Column '{atrial_col}' (ATRIAL_RATE_SOURCE="
                          f"{ATRIAL_RATE_SOURCE!r}) not found in CSV")
    out["atrial_rate"] = pd.to_numeric(df[atrial_col], errors="coerce")

    out["pr_interval"] = pd.to_numeric(df[PR_INTERVAL_COL], errors="coerce")
    out["qrs_duration"] = pd.to_numeric(df[QRS_DURATION_COL], errors="coerce")
    out["qt_corrected"] = pd.to_numeric(df[QT_CORRECTED_COL], errors="coerce")

    age = pd.to_numeric(df[AGE_AT_ECG_COL], errors="coerce")
    out["age_at_ecg"] = age.clip(upper=AGE_CAP)

    return out[FEATURE_COLS]


# ----------------------------------------------------------------------
def compute_tabular_stats(df_stats_raw: pd.DataFrame) -> dict:
    """
    df_stats_raw: output of extract_raw_features() for the STATS_SPLIT rows
                  (raw, i.e. NOT yet imputed/standardized).
    Returns medians (for MEDIAN_IMPUTE_COLS), and mean/std of each
    CONTINUOUS_COLS AFTER imputation (matching what will actually be
    standardized at transform time), plus the sex mode as a fallback for
    unrecognized/missing sex values.
    """
    medians = {}
    for col in MEDIAN_IMPUTE_COLS:
        med = df_stats_raw[col].median(skipna=True)
        if pd.isna(med):
            raise ValueError(f"Cannot compute median for '{col}': all values "
                              f"missing in stats split '{STATS_SPLIT}'")
        medians[col] = float(med)

    sex_mode_series = df_stats_raw["sex"].mode(dropna=True)
    sex_mode = float(sex_mode_series.iloc[0]) if len(sex_mode_series) else 0.0

    # Build the imputed (but not yet standardized) version to get mean/std
    imputed = df_stats_raw.copy()
    imputed["sex"] = imputed["sex"].fillna(sex_mode)
    for col in ZERO_IMPUTE_COLS:
        imputed[col] = imputed[col].fillna(0.0)
    for col in MEDIAN_IMPUTE_COLS:
        imputed[col] = imputed[col].fillna(medians[col])

    means = {}
    stds = {}
    for col in CONTINUOUS_COLS:
        means[col] = float(imputed[col].mean())
        std = float(imputed[col].std(ddof=0))
        stds[col] = std if std > 0 else 1.0

    stats = {
        "feature_cols": FEATURE_COLS,
        "continuous_cols": CONTINUOUS_COLS,
        "zero_impute_cols": ZERO_IMPUTE_COLS,
        "median_impute_cols": MEDIAN_IMPUTE_COLS,
        "medians": medians,
        "sex_mode": sex_mode,
        "mean": means,
        "std": stds,
        "age_cap": AGE_CAP,
        "atrial_rate_source": ATRIAL_RATE_SOURCE,
        "stats_split": STATS_SPLIT,
        "n_records_used": len(df_stats_raw),
    }
    return stats


def save_stats(stats: dict, path: str):
    with open(path, "w") as f:
        json.dump(stats, f, indent=2)
    print(f"[DONE] tabular stats saved -> {path}")


def load_stats(path: str) -> dict:
    with open(path, "r") as f:
        return json.load(f)

# ----------------------------------------------------------------------
def transform_features(df_raw: pd.DataFrame, stats: dict) -> np.ndarray:
    """
    df_raw: output of extract_raw_features() for any split.
    Applies imputation, sex binarization (already done in extract step, just
    fills missing), and standardization using the saved stats.
    Returns an (N, 7) float32 array in FEATURE_COLS order.
    """
    df = df_raw.copy()

    df["sex"] = df["sex"].fillna(stats["sex_mode"])
    for col in ZERO_IMPUTE_COLS:
        df[col] = df[col].fillna(0.0)
    for col in MEDIAN_IMPUTE_COLS:
        df[col] = df[col].fillna(stats["medians"][col])

    for col in CONTINUOUS_COLS:
        mean = stats["mean"][col]
        std = stats["std"][col]
        df[col] = (df[col] - mean) / std

    return df[FEATURE_COLS].to_numpy(dtype=np.float32)

# ----------------------------------------------------------------------
df = pd.read_csv(CSV_PATH)

if SPLIT_COL not in df.columns:
    raise ValueError(f"Expected a '{SPLIT_COL}' column in {CSV_PATH}")

unknown = set(df[SPLIT_COL].unique()) - set(SPLIT_ORDER)
if unknown:
    raise ValueError(f"Unexpected values in '{SPLIT_COL}': {unknown}. "
                        f"Expected a subset of {SPLIT_ORDER}")

# Re-assert split-block ordering (idempotent/no-op if the CSV is already
# sorted, e.g. because it's the waveform script's own output).
df[SPLIT_COL] = pd.Categorical(df[SPLIT_COL], categories=SPLIT_ORDER, ordered=True)
df = df.sort_values(SPLIT_COL, kind="stable").reset_index(drop=True)
df[SPLIT_COL] = df[SPLIT_COL].astype(str)

splits_present = [s for s in SPLIT_ORDER if s in df[SPLIT_COL].unique()]

raw_features = extract_raw_features(df)
raw_features[SPLIT_COL] = df[SPLIT_COL].values

# ---- Stats, computed once from STATS_SPLIT ----
if COMPUTE_STATS:
    if STATS_SPLIT not in splits_present:
        raise ValueError(
            f"STATS_SPLIT='{STATS_SPLIT}' not found among splits {splits_present}"
        )
    raw_stats_split = raw_features[raw_features[SPLIT_COL] == STATS_SPLIT]
    stats = compute_tabular_stats(raw_stats_split.drop(columns=[SPLIT_COL]))
    save_stats(stats, STATS_PATH)
elif os.path.exists(STATS_PATH):
    stats = load_stats(STATS_PATH)
    print(f"[INFO] loaded existing stats from {STATS_PATH}")
else:
    raise FileNotFoundError(
        f"COMPUTE_STATS=False, but {STATS_PATH} does not exist. Either "
        f"set COMPUTE_STATS=True once, or point STATS_PATH at an "
        f"existing stats file."
    )

# ---- Build each split's tabular_features.npy ----
for split_name in splits_present:
    df_split_raw = raw_features[raw_features[SPLIT_COL] == split_name].drop(columns=[SPLIT_COL])
    arr = transform_features(df_split_raw, stats)

    out_path = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX}_{split_name}_tabular_features.npy")
    np.save(out_path, arr)

    print(f"[DONE] {split_name}: {out_path} shape={arr.shape}")


/tmp/ipykernel_367214/1081011075.py:209: DtypeWarning: Columns (48,49,50,55) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


[DONE] tabular stats saved -> /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_tabular_stats.json
[DONE] train: /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_train_tabular_features.npy shape=(176860, 7)
[DONE] val: /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_val_tabular_features.npy shape=(10886, 7)
[DONE] test: /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_test_tabular_features.npy shape=(10887, 7)
[DONE] no_split: /media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_no_split_tabular_features.npy shape=(95935, 7)


In [25]:
CSV_PATH

'MIMIC-IV-Echonext_final.csv'

In [26]:
df = pd.read_csv(CSV_PATH)
#Rename columns if found
df = df.rename(columns = {
'lvef_lt_45_flag': 'lvef_lte_45_flag',
'pericardial_effusion_moderate_greater_flag': 'pericardial_effusion_moderate_large_flag',
'pasp_gt_45_flag': 'pasp_gte_45_flag'
})
#Save in the same path
df.to_csv('/media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_metadata.csv')

/tmp/ipykernel_367214/2897571825.py:1: DtypeWarning: Columns (47,49,54) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


## Sanity check for label NaN and prevalence

In [22]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# LABELS
# ============================================================

LABEL_COLS = [
    "lvef_lte_45_flag",
    "lvwt_gte_13_flag",
    "aortic_stenosis_moderate_or_greater_flag",
    "aortic_regurgitation_moderate_or_greater_flag",
    "mitral_regurgitation_moderate_or_greater_flag",
    "tricuspid_regurgitation_moderate_or_greater_flag",
    "pulmonary_regurgitation_moderate_or_greater_flag",
    "rv_systolic_dysfunction_moderate_or_greater_flag",
    "pericardial_effusion_moderate_large_flag",
    "pasp_gte_45_flag",
    "tr_max_gte_32_flag",
    "shd_moderate_or_greater_flag",
]


LABEL_NAMES = {
    "lvef_lte_45_flag":
        "LVEF ≤45%",

    "lvwt_gte_13_flag":
        "LV wall thickness ≥1.3 cm",

    "aortic_stenosis_moderate_or_greater_flag":
        "Moderate or greater aortic stenosis",

    "aortic_regurgitation_moderate_or_greater_flag":
        "Moderate or greater aortic regurgitation",

    "mitral_regurgitation_moderate_or_greater_flag":
        "Moderate or greater mitral regurgitation",

    "tricuspid_regurgitation_moderate_or_greater_flag":
        "Moderate or greater tricuspid regurgitation",

    "pulmonary_regurgitation_moderate_or_greater_flag":
        "Moderate or greater pulmonary regurgitation",

    "rv_systolic_dysfunction_moderate_or_greater_flag":
        "Moderate or greater RV systolic dysfunction",

    "pericardial_effusion_moderate_large_flag":
        "Moderate/large pericardial effusion",

    "pasp_gte_45_flag":
        "PASP ≥45 mmHg",

    "tr_max_gte_32_flag":
        "TR maximum velocity ≥3.2 m/s",

    "shd_moderate_or_greater_flag":
        "Any moderate or greater structural heart disease",
}


# ============================================================
# MAIN FUNCTION
# ============================================================

def inspect_labels(
    csv_path,
    labels_npy_path=None,
    split=None,
    save_table_path=None,
):
    """
    Inspect label missingness and prevalence in a metadata CSV.

    Optionally compare against an exported NumPy label array to check
    whether NaNs were preserved or were imputed somewhere downstream.

    Parameters
    ----------
    csv_path : str or Path
        Path to metadata CSV containing label columns.

    labels_npy_path : str or Path, optional
        Path to exported label .npy.
        Expected shape: (N, number_of_labels).

    split : str, optional
        If provided and CSV contains a 'split' column, analyze only
        this split, e.g. 'train', 'val', or 'test'.

    save_table_path : str or Path, optional
        Save manuscript table to CSV.

    Returns
    -------
    summary : pd.DataFrame
        Label availability and prevalence table.
    """

    csv_path = Path(csv_path)

    df = pd.read_csv(csv_path)

    # --------------------------------------------------------
    # Check expected columns
    # --------------------------------------------------------

    missing_columns = [
        col for col in LABEL_COLS
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Missing label columns in CSV:\n"
            + "\n".join(missing_columns)
        )

    # --------------------------------------------------------
    # Optional split filtering
    # --------------------------------------------------------

    if split is not None:

        if "split" not in df.columns:
            raise ValueError(
                "split was provided, but CSV has no 'split' column."
            )

        df = (
            df[
                df["split"]
                .astype(str)
                .str.lower()
                .eq(split.lower())
            ]
            .reset_index(drop=True)
        )

    print("=" * 70)
    print("CSV LABEL INSPECTION")
    print("=" * 70)

    print(f"CSV: {csv_path}")
    print(f"N rows analyzed: {len(df):,}")

    if split is not None:
        print(f"Split: {split}")

    # Keep float so NaNs survive
    y_csv = df[LABEL_COLS].astype(float).to_numpy()

    print(
        f"\nAny NaN in CSV labels? "
        f"{np.isnan(y_csv).any()}"
    )

    print(
        f"Total NaN label cells: "
        f"{np.isnan(y_csv).sum():,}"
    )

    print(
        f"Total label cells: "
        f"{y_csv.size:,}"
    )

    print(
        f"Overall observed-label fraction: "
        f"{np.isfinite(y_csv).mean():.4%}"
    )

    # --------------------------------------------------------
    # Summary by class
    # --------------------------------------------------------

    rows = []

    for col in LABEL_COLS:

        values = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        known = values.notna()

        n_total = len(values)
        n_known = int(known.sum())
        n_unknown = int((~known).sum())

        n_positive = int((values == 1).sum())
        n_negative = int((values == 0).sum())

        # Prevalence is calculated ONLY among known labels
        prevalence = (
            n_positive / n_known
            if n_known > 0
            else np.nan
        )

        known_pct = (
            100 * n_known / n_total
            if n_total > 0
            else np.nan
        )

        unknown_pct = (
            100 * n_unknown / n_total
            if n_total > 0
            else np.nan
        )

        rows.append({
            "Outcome": LABEL_NAMES.get(col, col),
            "Variable": col,

            "Total N": n_total,

            "Known N": n_known,
            "Known (%)": known_pct,

            "Unknown N": n_unknown,
            "Unknown (%)": unknown_pct,

            "Positive N": n_positive,
            "Negative N": n_negative,

            "Prevalence among known (%)":
                100 * prevalence
                if np.isfinite(prevalence)
                else np.nan,
        })

    summary = pd.DataFrame(rows)

    # --------------------------------------------------------
    # Validate values
    # --------------------------------------------------------

    print("\nUnique non-missing label values:")

    for col in LABEL_COLS:

        vals = sorted(
            pd.to_numeric(
                df[col],
                errors="coerce"
            )
            .dropna()
            .unique()
            .tolist()
        )

        print(f"{col}: {vals}")

        unexpected = [
            x for x in vals
            if x not in [0, 1]
        ]

        if unexpected:
            print(
                f"  WARNING: unexpected values: {unexpected}"
            )

    # ========================================================
    # OPTIONAL NUMPY COMPARISON
    # ========================================================

    if labels_npy_path is not None:

        labels_npy_path = Path(labels_npy_path)

        y_npy = np.load(
            labels_npy_path,
            allow_pickle=False
        )

        print("\n" + "=" * 70)
        print("NUMPY LABEL INSPECTION")
        print("=" * 70)

        print(f"NPY: {labels_npy_path}")
        print(f"Shape: {y_npy.shape}")
        print(f"dtype: {y_npy.dtype}")

        print(
            f"Any NaN in NPY? "
            f"{np.issubdtype(y_npy.dtype, np.floating) and np.isnan(y_npy).any()}"
        )

        if np.issubdtype(
            y_npy.dtype,
            np.floating
        ):
            npy_nan_count = int(
                np.isnan(y_npy).sum()
            )
        else:
            npy_nan_count = 0

        csv_nan_count = int(
            np.isnan(y_csv).sum()
        )

        print(f"NaNs in CSV: {csv_nan_count:,}"  )

        print(            f"NaNs in NPY: {npy_nan_count:,}"        )

        # ----------------------------------------------------
        # Shape comparison
        # ----------------------------------------------------

        expected_shape = (
            len(df),
            len(LABEL_COLS)
        )

        if y_npy.shape != expected_shape:

            print(
                "\nWARNING: NPY shape does not match "
                "the analyzed CSV label matrix."
            )

            print(
                f"Expected: {expected_shape}"
            )

            print(
                f"Observed: {y_npy.shape}"
            )

        else:

            # ------------------------------------------------
            # Determine whether missing labels became zeros
            # ------------------------------------------------

            csv_missing = np.isnan(y_csv)

            if csv_missing.any():

                exported_at_missing_positions = (
                    y_npy[csv_missing]
                )

                n_became_zero = int(
                    np.sum(
                        exported_at_missing_positions == 0
                    )
                )

                n_became_one = int(
                    np.sum(
                        exported_at_missing_positions == 1
                    )
                )

                if np.issubdtype(
                    y_npy.dtype,
                    np.floating
                ):

                    n_remained_nan = int(
                        np.isnan(
                            exported_at_missing_positions
                        ).sum()
                    )

                else:
                    n_remained_nan = 0

                print(
                    "\nWhat happened to CSV NaNs "
                    "at the corresponding NPY locations?"
                )

                print(
                    f"Remained NaN: {n_remained_nan:,}"
                )

                print(
                    f"Became 0:     {n_became_zero:,}"
                )

                print(
                    f"Became 1:     {n_became_one:,}"
                )

                if (
                    csv_nan_count > 0
                    and n_became_zero == csv_nan_count
                ):

                    print(
                        "\n*** IMPORTANT ***"
                    )

                    print(
                        "Every missing CSV label became 0 "
                        "in the NPY."
                    )

                    print(
                        "This strongly indicates that labels "
                        "were imputed as negative before or "
                        "during export."
                    )

                elif (
                    csv_nan_count > 0
                    and n_remained_nan == csv_nan_count
                ):

                    print(
                        "\nGood: all missing labels were "
                        "preserved as NaN."
                    )

                else:

                    print(
                        "\nMixed behavior detected. Inspect "
                        "the label-generation/export pipeline."
                    )

            else:

                print(
                    "\nThe CSV itself contains no NaN target "
                    "labels."
                )

                print(
                    "Therefore, if missing measurements existed "
                    "originally, they were resolved/imputed "
                    "UPSTREAM of this CSV."
                )

    # --------------------------------------------------------
    # Pretty manuscript table
    # --------------------------------------------------------

    manuscript_table = summary[[
        "Outcome",
        "Known N",
        "Known (%)",
        "Unknown N",
        "Unknown (%)",
        "Positive N",
        "Negative N",
        "Prevalence among known (%)",
    ]].copy()

    # Formatting only
    for col in [
        "Known (%)",
        "Unknown (%)",
        "Prevalence among known (%)",
    ]:
        manuscript_table[col] = (
            manuscript_table[col]
            .round(1)
        )

    print("\n" + "=" * 70)
    print("MANUSCRIPT TABLE")
    print("=" * 70)

    print(
        manuscript_table.to_string(
            index=False
        )
    )

    if save_table_path is not None:

        save_table_path = Path(
            save_table_path
        )

        manuscript_table.to_csv(
            save_table_path,
            index=False
        )

        print(
            f"\nSaved manuscript table to:"
            f"\n{save_table_path}"
        )

    return summary
summary = inspect_labels(
    csv_path="/media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_metadata.csv",  # output of the waveform script
    save_table_path="/media/amendoza/HDD/MIMICIV_ECG_echo/MIMICIV_ECG_Echo_label_availability_table.csv",
)



/tmp/ipykernel_367214/995139561.py:106: DtypeWarning: Columns (46,47,48,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


ValueError: Missing label columns in CSV:
lvef_lte_45_flag
pericardial_effusion_moderate_large_flag
pasp_gte_45_flag